# Phase 5 — ReAct Agent with 5 Tools

## Purpose
This notebook demonstrates **Phase 5** of the AI-Powered Loan Origination Copilot: replacing the Phase 4 single-shot LLM call with a full **ReAct agent loop** that chains 5 deterministic tools automatically.

## What is a ReAct Agent?
ReAct = **Re**ason + **Act**. Instead of generating one response, the agent runs a loop:

```
[Reason]  I need to check if this customer is eligible.
[Act]     → call check_eligibility(...)
[Observe] ← FOIR 34% — PASS. Rate: 8.5%–9.5%

[Reason]  Eligible. Now calculate the EMI range.
[Act]     → call calculate_emi(principal=3000000, rate=8.5, tenure=240)
[Observe] ← EMI ₹26,085/month

[Reason]  Now I can compile the document checklist.
[Act]     → call get_document_checklist(home_loan, salaried)
[Observe] ← [salary slips, Form 16, property documents, ...]

[Answer]  Compose complete response with all three results.
```

This loop is powered by **OpenAI function calling** (GPT-4o's native tool API), not text parsing. The LLM returns a structured tool-call JSON; the agent framework executes it and feeds the result back as context for the next reasoning step.

## Demos in this notebook
| # | Demo | Tools called |
|---|------|--------------|
| 1 | Eligible customer — full Q1 flow | eligibility → EMI → document checklist |
| 2 | Ineligible customer — FOIR breach | eligibility only (stops here) |
| 3 | Escalation — MSME ₹5 Cr above ceiling | eligibility → escalation summary |
| 4 | FAQ query — policy RAG via lookup tool | lookup_loan_status |
| 5 | Multi-turn memory — no field re-ask | all tools, across 2 turns |
| 6 | Langfuse trace verification | all tool calls visible as structured spans |

## What Phase 5 adds over Phase 4
| Dimension | Phase 4 (RAG only) | Phase 5 (ReAct + tools) |
|-----------|-------------------|--------------------------|
| EMI calculation | LLM estimates (varies!) | Exact reducing-balance formula |
| Eligibility | LLM guesses | Deterministic FOIR + CIBIL rules |
| Document checklist | RAG-retrieved text | Exact lookup table |
| Escalation | Never auto-triggered | Auto-triggers when amount > ceiling |
| Policy FAQ | RAG context injected | `lookup_loan_status` tool called on demand |
| Langfuse trace | One LLM span | Multiple tool spans + LLM reasoning steps |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SETUP — Run this cell first. Safe to re-run.
# Works in Google Colab and local Jupyter equally.
# ─────────────────────────────────────────────────────────────────────────────
import sys, subprocess, os

# 1. Install required packages
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'langchain', 'langchain-openai', 'langchain-community', 'langchain-core',
    'langchain-chroma', 'langchain-text-splitters',
    'python-dotenv', 'chromadb', 'openai', 'tiktoken', 'langfuse'],
    check=True)
print('✓ Packages ready')

# 2. Detect runtime and set project root
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Capstone/code'
else:
    PROJECT_ROOT = os.path.abspath('..')

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f'✓ Project root: {PROJECT_ROOT}')

# 3. Load keys
from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, '.env'))

if IN_COLAB:
    from google.colab import userdata
    for key in ['OPENAI_API_KEY', 'OPENAI_BASE_URL',
                'LANGFUSE_PUBLIC_KEY', 'LANGFUSE_SECRET_KEY',
                'LANGFUSE_HOST', 'LANGFUSE_BASE_URL']:
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
        except Exception:
            pass

if not os.environ.get('OPENAI_API_KEY'):
    import getpass
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY not found — enter it now: ')

print('✓ Environment ready')
print(f'  OPENAI_API_KEY : set')
print(f'  Langfuse keys  : {"set" if os.environ.get("LANGFUSE_PUBLIC_KEY") else "not set (optional)"}')

---
## Tool Registry
Phase 5 registers 5 tools with the agent. Each tool has a typed schema — the LLM reads these descriptions to decide when and how to call each one.

In [ ]:
from tools.tool_registry import get_all_tools

tools = get_all_tools()
print(f'Registered tools: {len(tools)}')
print()
for t in tools:
    print(f'  {t.name}')
    # First line of the docstring = the tool's purpose (what the LLM reads)
    first_line = t.description.strip().split('\n')[0]
    print(f'    → {first_line}')

---
## Demo 1 — Eligible Customer: Full Q1 Flow

**Input:** Complete customer profile — all 6 fields in one message.  
**Expected tool chain:** `check_eligibility` → `calculate_emi` → `get_document_checklist`  
**Key check:** EMI figure is exact (reducing-balance formula), not an LLM estimate.

In [ ]:
from agent.core_agent import LoanCopilotAgent

agent = LoanCopilotAgent(phase=5)
print(f'Phase 5 agent ready. Session: {agent.session_id}')
print()

q1 = (
    'I am 35 years old, salaried, earning Rs 80,000 per month, CIBIL 740. '
    'I want a home loan of 30 lakhs for 20 years.'
)
print(f'Query: {q1}')
print()
response = agent.chat(q1)
print('=== Agent Response ===')
print(response)
print()

# Verify EMI against the deterministic formula
from tools.emi_calculator import calculate_emi
emi_lo = calculate_emi.invoke({'principal': 3000000, 'annual_rate_percent': 8.5, 'tenure_months': 240})
emi_hi = calculate_emi.invoke({'principal': 3000000, 'annual_rate_percent': 9.5, 'tenure_months': 240})
print(f'Deterministic EMI range: ₹{emi_lo["emi"]:,.0f} – ₹{emi_hi["emi"]:,.0f}/month')
print('NOTE: Phase 5 agent EMI should match these exact figures.')

---
## Demo 2 — Ineligible Customer: FOIR Breach

**Input:** Same income, but loan amount too large → FOIR > 50%.  
**Expected tool chain:** `check_eligibility` only (agent stops — no EMI or docs needed).  
**Key check:** Agent explains the FOIR breach clearly and suggests remedies.

In [ ]:
agent2 = LoanCopilotAgent(phase=5)

q_ineligible = (
    'I am 35 years old, salaried, earning Rs 80,000 per month, CIBIL 740. '
    'I want a home loan of 50 lakhs for 20 years.'
)
print(f'Query: {q_ineligible}')
print()
response = agent2.chat(q_ineligible)
print('=== Agent Response ===')
print(response)
print()

# Verify: FOIR for 50L at mid-rate
emi_mid = calculate_emi.invoke({'principal': 5000000, 'annual_rate_percent': 9.0, 'tenure_months': 240})
foir = emi_mid['emi'] / 80000
print(f'Calculated FOIR: {foir:.1%}  (limit 50% for income ≤ ₹1L) — correctly REJECTED')

---
## Demo 3 — Escalation: MSME Loan Above Advisory Ceiling

**Input:** MSME loan of ₹5 Cr — above the ₹2 Cr advisory ceiling.  
**Expected tool chain:** `check_eligibility` → `generate_escalation_summary`  
**Key check:** Agent invokes escalation tool automatically and tells customer an RM will follow up.

In [ ]:
from deployment.config import ESCALATION_CEILINGS

agent3 = LoanCopilotAgent(phase=5)
print(f'MSME escalation ceiling: ₹{ESCALATION_CEILINGS["msme_loan"]/1e7:.0f} Cr')
print()

q_escalate = (
    'I run a manufacturing business for 8 years. Monthly income 5 lakh. '
    'I need an MSME loan of 5 crore for 10 years. Age 45, CIBIL 730.'
)
print(f'Query: {q_escalate}')
print()
response = agent3.chat(q_escalate)
print('=== Agent Response ===')
print(response)

---
## Demo 4 — Policy FAQ via RAG Tool

**Input:** A policy question (NRI eligibility, prepayment charges) — no profile needed.  
**Expected tool:** `lookup_loan_status` — agent calls it to retrieve policy context from ChromaDB.  
**Key check:** Answer comes from the policy document, not LLM training data.

In [ ]:
agent4 = LoanCopilotAgent(phase=5)

faq_queries = [
    'Can an NRI apply for a home loan?',
    'Are there prepayment charges on a car loan?',
]

for q in faq_queries:
    agent4.reset()
    r = agent4.chat(q)
    print(f'Q: {q}')
    print(f'A: {r}')
    print()

---
## Demo 5 — Multi-Turn Memory: No Field Re-Ask

Turn 1: Customer provides income + product.  
Turn 2: Customer provides remaining details.  
**Key check:** Agent does not re-ask for income or product in Turn 2 — memory preserved across turns.

In [ ]:
agent5 = LoanCopilotAgent(phase=5)

turn1 = 'I want a home loan. I earn Rs 1.2 lakh per month and I am salaried.'
turn2 = 'I am 40 years old, CIBIL score is 760. I need 60 lakhs for 15 years.'

print('=== Turn 1 ===')
print(f'User: {turn1}')
r1 = agent5.chat(turn1)
print(f'Agent: {r1}')
print()

print('=== Turn 2 ===')
print(f'User: {turn2}')
r2 = agent5.chat(turn2)
print(f'Agent: {r2}')
print()

print(f'Memory size after 2 turns: {len(agent5.memory.chat_memory.messages)} messages')
print('(income + product from Turn 1 carried into Turn 2 context — no re-ask)')

---
## Demo 6 — Langfuse Trace Verification

Each `agent.chat()` call in Phase 5 generates a Langfuse trace with **multiple spans** — one per tool call plus the LLM reasoning steps.

In `cloud.langfuse.com → Traces`, open a Phase 5 trace and expand it. You should see:
- An outer agent span (the full turn)
- `check_eligibility` tool span with input args and return dict
- `calculate_emi` tool span (if eligible)
- `get_document_checklist` tool span (if eligible)
- Or `generate_escalation_summary` span (if escalated)

This multi-span structure is the key difference from Phase 3/4 which had a single LLM span.

In [ ]:
from deployment.config import LANGFUSE_PUBLIC_KEY, LANGFUSE_HOST
from monitoring.langfuse_logger import flush

if LANGFUSE_PUBLIC_KEY:
    flush()
    # Re-run the eligible case so the trace is fresh
    trace_agent = LoanCopilotAgent(phase=5)
    trace_agent.chat(
        'I am 35 years old, salaried, Rs 80K income, CIBIL 750. '
        'Home loan 30 lakhs for 20 years.'
    )
    flush()
    print(f'✓ Trace flushed → {LANGFUSE_HOST}')
    print(f'  Session ID: {trace_agent.session_id}')
    print()
    print('In cloud.langfuse.com → Traces → open this session to see:')
    print('  • check_eligibility  — input: all 8 args | output: eligible=True, foir=...')
    print('  • calculate_emi      — input: principal, rate, tenure | output: emi=...')
    print('  • get_document_checklist — input: product, employment | output: doc lists')
else:
    print('Langfuse not configured. Add keys to Colab Secrets or .env to enable tracing.')

---
## Phase 5 Limitations

These motivate Phase 6 (persistent multi-turn memory) and Phase 7 (adaptive behaviour).

### Limitation 1 — Profile is not persisted across sessions
Each `LoanCopilotAgent` instance has its own in-memory `SessionState`. If the customer closes the browser and returns, the profile is gone. Phase 6 wires `ConversationBufferWindowMemory` to a persistent store so sessions can be resumed.

### Limitation 2 — No feedback loop
The agent has no mechanism to learn from thumbs-up/thumbs-down feedback. Phase 7 (RLHF) records ratings and uses them to adapt the tone (empathy injection when avg rating < 0.6).

### Limitation 3 — Safety gate is not layered in front of the agent
Phase 5 relies solely on the system prompt to block out-of-scope inputs. Phase 8 adds a two-stage gate (keyword filter + GPT-4o-mini intent classifier) that intercepts queries before they reach the agent.